In [52]:
!apt-get install -y libzbar0

!pip install pyzbar
!pip install Pillow
!pip install langgraph
!pip install -U langchain-openai
!pip install openpyxl
!pip install pymupdf
!pip install beautifulsoup4

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libzbar0 is already the newest version (0.23.92-4build2).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [53]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
import os
import time
import fitz
import requests
import pandas as pd

from PIL import Image
from pyzbar.pyzbar import decode
from bs4 import BeautifulSoup

from typing import TypedDict, List

from google.colab import userdata

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI




In [55]:
class AgentState(TypedDict):

    cv_text: str

    folder_path: str

    extracted_urls: List[str]

    job_details: List[dict]

    final_report: List[dict]

In [56]:
def extract_cv_text(pdf_path):

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text

In [57]:
from PIL import Image
from pyzbar.pyzbar import decode
import os

folder = "/content/drive/MyDrive/barcodes"

for file in os.listdir(folder):

    if file.endswith((".png", ".jpg", ".jpeg")):

        path = os.path.join(folder, file)

        img = Image.open(path)

        result = decode(img)

        print(file)
        print(result)
        print("-" * 50)

photo_5953998246759828661_y.jpg
[Decoded(data=b'ksahr@carrier.com', type='QRCODE', rect=Rect(left=336, top=387, width=248, height=223), polygon=[Point(x=336, y=414), Point(x=367, y=610), Point(x=584, y=575), Point(x=537, y=387)], quality=1, orientation='UP')]
--------------------------------------------------
photo_5953998246759828662_y.jpg
[Decoded(data=b'https://forms.office.com/r/m112x3FN5i?origin=QRCode&qrcodeorigin=presentation', type='QRCODE', rect=Rect(left=253, top=356, width=427, height=329), polygon=[Point(x=253, y=361), Point(x=284, y=685), Point(x=617, y=670), Point(x=680, y=356)], quality=1, orientation='UP')]
--------------------------------------------------
photo_5953998246759828663_y.jpg
[]
--------------------------------------------------
photo_5953998246759828664_y.jpg
[]
--------------------------------------------------
photo_5953998246759828665_y.jpg
[Decoded(data=b'https://efan.fa.em3.oraclecloud.com/hcmUI/CandidateExperience/en/sites/CX_2001/job/2474', type='QR

In [58]:
def scrape_jobs_node(state: AgentState):

    print(
        f"--- Scraping {len(state['extracted_urls'])} jobs ---"
    )

    jobs_info = []

    for url in state["extracted_urls"]:

        try:

            response = requests.get(
                url,
                timeout=10,
                headers={
                    "User-Agent":
                    "Mozilla/5.0"
                }
            )

            response.raise_for_status()

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            for tag in soup(
                ["script", "style", "noscript"]
            ):
                tag.decompose()

            text = soup.get_text(
                separator=" ",
                strip=True
            )

            jobs_info.append({
                "url": url,
                "content": text[:3000]
            })

        except Exception as e:

            print(
                f"Failed: {url}\n{e}"
            )

    return {
        "job_details": jobs_info
    }

In [59]:
def match_evaluator_node(state: AgentState):

    print("--- Matching Jobs ---")

    openrouter_key = userdata.get("OpenRouter")

    llm = ChatOpenAI(
        model="openai/gpt-3.5-turbo",
        openai_api_key=openrouter_key,
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0
    )

    cv_text = state["cv_text"]

    results = []

    for job in state["job_details"]:

        try:

            prompt = f"""
Compare this CV:

{cv_text[:4000]}

With this job description:

{job['content'][:3000]}

Return ONLY in this format:

Score: <number from 0 to 100>

Strengths: <comma separated list>

Missing Skills: <comma separated list>

Job Title: <job title>

Summary: <one sentence>
"""

            response = llm.invoke(prompt)

            text = response.content

            score = ""
            strengths = ""
            missing_skills = ""
            summary = ""
            job_title = ""

            for line in text.split("\n"):

                line = line.strip()

                if line.startswith("Score:"):
                    score = line.replace(
                        "Score:", ""
                    ).strip()

                elif line.startswith("Strengths:"):
                    strengths = line.replace(
                        "Strengths:", ""
                    ).strip()

                elif line.startswith("Missing Skills:"):
                    missing_skills = line.replace(
                        "Missing Skills:", ""
                    ).strip()

                elif line.startswith("Job Title:"):
                    job_title = line.replace(
                        "Job Title:", ""
                    ).strip()

                elif line.startswith("Summary:"):
                    summary = line.replace(
                        "Summary:", ""
                    ).strip()

            results.append({
                "url": job["url"],
                "job_title": job_title,
                "score": score,
                "strengths": strengths,
                "missing_skills": missing_skills,
                "summary": summary
            })

            time.sleep(1)

        except Exception as e:

            print(f"Error: {e}")

    return {
        "final_report": results
    }

In [60]:
workflow = StateGraph(
    AgentState
)

workflow.add_node(
    "extractor",
    extract_barcodes_node
)

workflow.add_node(
    "scraper",
    scrape_jobs_node
)

workflow.add_node(
    "matcher",
    match_evaluator_node
)

workflow.set_entry_point(
    "extractor"
)

workflow.add_edge(
    "extractor",
    "scraper"
)

workflow.add_edge(
    "scraper",
    "matcher"
)

workflow.add_edge(
    "matcher",
    END
)

app = workflow.compile()

In [61]:
cv_path = "/content/drive/MyDrive/Areej-Alzaidi-CV(Update).pdf.pdf"

cv_text = extract_cv_text(
    cv_path
)

In [62]:
result = app.invoke(inputs)

--- Extracting URLs ---
--- Scraping 0 jobs ---
--- Matching Jobs ---


In [63]:
inputs = {
    "folder_path":
    "/content/drive/MyDrive/barcodes",

    "cv_text":
    cv_text
}

result = app.invoke(inputs)

--- Extracting URLs ---
--- Scraping 33 jobs ---
Failed: Recruitment@baeshen.com
Invalid URL 'Recruitment@baeshen.com': No scheme supplied. Perhaps you meant https://Recruitment@baeshen.com?
Failed: https://qrfy.io/QiyL61_kWF
404 Client Error: Not Found for url: https://qrfy.io/qr/invalid
Failed: ksahr@carrier.com
Invalid URL 'ksahr@carrier.com': No scheme supplied. Perhaps you meant https://ksahr@carrier.com?
Failed: https://sysdown.sawaeed.sa/companySurvey/contact/home
HTTPSConnectionPool(host='sysdown.sawaeed.sa', port=443): Max retries exceeded with url: /companySurvey/contact/home (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f7a46d65190>, 'Connection to sysdown.sawaeed.sa timed out. (connect timeout=10)'))
Failed: https://forms.gle/1LYE49kPjfh8qysr7
401 Client Error: Unauthorized for url: https://docs.google.com/forms/d/e/1FAIpQLSdXu8ClQ3r4_vh3miIGhUemLcj4Ggs6uClmlbHON6_jr7UaaQ/viewform?usp=send_form
Failed: https://career.ncms.sa/
HTTPSConnectio

In [64]:
inputs = {
    "folder_path": "/content/drive/MyDrive/barcodes",
    "cv_text": cv_text
}

result = app.invoke(inputs)

print(result)

--- Extracting URLs ---
--- Scraping 33 jobs ---
Failed: Recruitment@baeshen.com
Invalid URL 'Recruitment@baeshen.com': No scheme supplied. Perhaps you meant https://Recruitment@baeshen.com?
Failed: https://qrfy.io/QiyL61_kWF
404 Client Error: Not Found for url: https://qrfy.io/qr/invalid
Failed: ksahr@carrier.com
Invalid URL 'ksahr@carrier.com': No scheme supplied. Perhaps you meant https://ksahr@carrier.com?
Failed: https://sysdown.sawaeed.sa/companySurvey/contact/home
HTTPSConnectionPool(host='sysdown.sawaeed.sa', port=443): Max retries exceeded with url: /companySurvey/contact/home (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f7a465893a0>, 'Connection to sysdown.sawaeed.sa timed out. (connect timeout=10)'))
Failed: https://forms.gle/1LYE49kPjfh8qysr7
401 Client Error: Unauthorized for url: https://docs.google.com/forms/d/e/1FAIpQLSdXu8ClQ3r4_vh3miIGhUemLcj4Ggs6uClmlbHON6_jr7UaaQ/viewform?usp=send_form
Failed: https://career.ncms.sa/
HTTPSConnectio

In [65]:
inputs = {
    "folder_path":
    "/content/drive/MyDrive/barcodes",

    "cv_text":
    cv_text
}

result = app.invoke(inputs)

--- Extracting URLs ---
--- Scraping 33 jobs ---
Failed: Recruitment@baeshen.com
Invalid URL 'Recruitment@baeshen.com': No scheme supplied. Perhaps you meant https://Recruitment@baeshen.com?
Failed: https://qrfy.io/QiyL61_kWF
404 Client Error: Not Found for url: https://qrfy.io/qr/invalid
Failed: ksahr@carrier.com
Invalid URL 'ksahr@carrier.com': No scheme supplied. Perhaps you meant https://ksahr@carrier.com?
Failed: https://sysdown.sawaeed.sa/companySurvey/contact/home
HTTPSConnectionPool(host='sysdown.sawaeed.sa', port=443): Max retries exceeded with url: /companySurvey/contact/home (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7f7a45c453a0>, 'Connection to sysdown.sawaeed.sa timed out. (connect timeout=10)'))
Failed: https://forms.gle/1LYE49kPjfh8qysr7
401 Client Error: Unauthorized for url: https://docs.google.com/forms/d/e/1FAIpQLSdXu8ClQ3r4_vh3miIGhUemLcj4Ggs6uClmlbHON6_jr7UaaQ/viewform?usp=send_form
Failed: https://career.ncms.sa/
HTTPSConnectio

In [66]:
print(result.keys())

print(
    "URLs:",
    len(result["extracted_urls"])
)

print(
    "Jobs:",
    len(result["job_details"])
)

print(
    "Reports:",
    len(result["final_report"])
)

dict_keys(['cv_text', 'folder_path', 'extracted_urls', 'job_details', 'final_report'])
URLs: 33
Jobs: 27
Reports: 27


In [67]:
report_df = pd.DataFrame(result["final_report"])

report_df.head()

,url,job_title,score,strengths,missing_skills,summary
0,https://forms.office.com/r/M90DhkgLY8?origin=Q...,"Cybersecurity Governance, Risk, and Compliance...",85,"Cybersecurity Risk Management, GRC, Regulatory...","Penetration Testing, Network Security, Inciden...",Experienced cybersecurity professional special...
1,https://forms.office.com/r/m112x3FN5i?origin=Q...,"Cybersecurity Governance, Risk, and Compliance...",85,"Cybersecurity Risk Management, GRC, Regulatory...","Penetration Testing, Network Security, Inciden...",Experienced cybersecurity professional special...
2,https://qrco.de/bgiqol,"Cybersecurity Governance, Risk, and Compliance...",85,"Cybersecurity Risk Management, GRC, Regulatory...","Penetration Testing, Network Security, Inciden...",Experienced cybersecurity professional special...
3,https://www.surveymonkey.com/r/qr_code/JDZLZ7M,Cybersecurity Analyst,85,"Cybersecurity Risk Management, Governance, Ris...","Penetration Testing, Network Security, Cloud S...",Experienced cybersecurity professional with a ...
4,https://alsalamahospital.com/,Cybersecurity Specialist,85,"Cybersecurity certifications, hands-on experie...","Healthcare industry knowledge, clinical experi...",Experienced cybersecurity professional with a ...


In [71]:
report_df["score"] = pd.to_numeric(
    report_df["score"],
    errors="coerce"
)
def get_match(score):

    if score >= 90:
        return "🟢 Excellent Match"

    elif score >= 80:
        return "🔵 Strong Match"

    elif score >= 70:
        return "🟡 Good Match"

    elif score >= 60:
        return "🟠 Fair Match"

    else:
        return "🔴 Weak Match"


report_df["match"] = report_df["score"].apply(get_match)

report_df = report_df.sort_values(
    by="score",
    ascending=False
)



,url,job_title,score,strengths,missing_skills,summary,match
0,https://forms.office.com/r/M90DhkgLY8?origin=Q...,"Cybersecurity Governance, Risk, and Compliance...",85,"Cybersecurity Risk Management, GRC, Regulatory...","Penetration Testing, Network Security, Inciden...",Experienced cybersecurity professional special...,🔵 Strong Match
1,https://forms.office.com/r/m112x3FN5i?origin=Q...,"Cybersecurity Governance, Risk, and Compliance...",85,"Cybersecurity Risk Management, GRC, Regulatory...","Penetration Testing, Network Security, Inciden...",Experienced cybersecurity professional special...,🔵 Strong Match
2,https://qrco.de/bgiqol,"Cybersecurity Governance, Risk, and Compliance...",85,"Cybersecurity Risk Management, GRC, Regulatory...","Penetration Testing, Network Security, Inciden...",Experienced cybersecurity professional special...,🔵 Strong Match
3,https://www.surveymonkey.com/r/qr_code/JDZLZ7M,Cybersecurity Analyst,85,"Cybersecurity Risk Management, Governance, Ris...","Penetration Testing, Network Security, Cloud S...",Experienced cybersecurity professional with a ...,🔵 Strong Match
4,https://alsalamahospital.com/,Cybersecurity Specialist,85,"Cybersecurity certifications, hands-on experie...","Healthcare industry knowledge, clinical experi...",Experienced cybersecurity professional with a ...,🔵 Strong Match


In [73]:
report_df[["job_title", "score", "match"]].head(10)

,job_title,score,match
0,"Cybersecurity Governance, Risk, and Compliance...",85,🔵 Strong Match
1,"Cybersecurity Governance, Risk, and Compliance...",85,🔵 Strong Match
2,"Cybersecurity Governance, Risk, and Compliance...",85,🔵 Strong Match
3,Cybersecurity Analyst,85,🔵 Strong Match
4,Cybersecurity Specialist,85,🔵 Strong Match
5,"Cybersecurity Governance, Risk, and Compliance...",85,🔵 Strong Match
6,"Cybersecurity Governance, Risk, and Compliance...",85,🔵 Strong Match
7,Cybersecurity Analyst,85,🔵 Strong Match
8,Cybersecurity Analyst,85,🔵 Strong Match
9,Cybersecurity Analyst,85,🔵 Strong Match


In [69]:
report_df.to_excel(
    "Job_Analysis_Report.xlsx",
    index=False
)

In [70]:
from google.colab import files

files.download(
    "Job_Analysis_Report.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>